# <center>**IROS**<center>

**Libraries**

In [1]:
from collections.abc import Sequence
from copy import deepcopy
from pathlib import Path
import numpy as np
from astropy.io import fits
from tqdm import tqdm
import pickle

from mbloodmoon.io import _validate_fits
from mbloodmoon.coords import shift2equatorial
from mbloodmoon.images import _shift
import mbloodmoon as bm

from iros_wrappers import perform_iros, save_pickle, load_pickle
from iros_wrappers import gen_log, computes_params
from iros_wrappers import save_iros_output, load_iros_output
from iros_wrappers import save_iros_data, load_iros_data

In [2]:
root_path = "/mnt/d/PhD_AASS/Coding/Images_fits/"
mask_file = root_path + "wfm_mask.fits"
simul_data = root_path + "iros_simulation_GC_LMC/20241011_galctr_rxte_sax_2-30keV_1ks_2cams_sources_cxb/"

cam_a = "cam1a"
cam_b = "cam1b"
dataset = "reconstructed"

filepaths = bm.simulation_files(simul_data)
wfm = bm.codedmask(mask_file, upscale_x=5, upscale_y=1)
sdlA = bm.simulation(filepaths[cam_a][dataset])
sdlB = bm.simulation(filepaths[cam_b][dataset])

max_iterations = 7
snr_threshold = 5

In [4]:
iros_output_name = "iros_output0.fits"

try:
    iros_output = load_iros_output(root_path + iros_output_name)
except FileNotFoundError:
    iros_output = perform_iros(
        wfm=wfm,
        sdlA=sdlA,
        sdlB=sdlB,
        cameras=(cam_a, cam_b),
        max_iterations=max_iterations,
        snr_threshold=snr_threshold,
    )
    save_iros_output(iros_output, mask_file, root_path + iros_output_name)

# Loading data...
# Loading completed!


In [5]:
iros_output

{'cam1a': {'shiftx': array([ 43.9241036 ,  13.55234697, -28.13559855,  53.60453744,
          29.87281134, -64.6       ,   8.44090305], dtype='>f8'),
  'shifty': array([ 78.37877528, -12.38659266,  29.15177642, -26.77957834,
         -13.16681316,  39.00123457,  -1.94002471], dtype='>f8'),
  'fluence': array([512698.88518584,  63210.85050942,  47467.67282463,  37951.14666424,
          30186.65655681,  21803.09959734,  19250.56352033], dtype='>f8'),
  'snr': array([604.71447068,  91.59016557,  68.0173082 ,  56.52162511,
          50.63000319,  36.71480434,  29.60654797], dtype='>f8'),
  'obs_counts': array([481648.49229751,  54646.5749376 ,  41227.02922032,  32944.53461653,
          29323.16285397,  19849.15310318,  17253.71073935], dtype='>f8'),
  'sub_counts': array([488088.85843708,  54699.88878246,  42941.82639921,  33154.86358414,
          28456.72165309,  19062.57471235,  17001.70314003], dtype='>f8')},
 'cam1b': {'shiftx': array([ 78.05241689, -12.48477313,  28.83757014, -27.1

In [6]:
iros_data_name = "iros_data0.fits"

try:
    iros_data = load_iros_data(
        root_path + iros_data_name,
        insert_header_info=True,
    )

except FileNotFoundError:
    save_to = root_path + iros_data_name
    log = gen_log((cam_a, cam_b))

    iros_data = computes_params(
        iros_output=iros_output,
        wfm=wfm,
        sdlA=sdlA,
        sdlB=sdlB,
        log=log,
        store_headers=True,
    )

    save_iros_data(
        data=iros_data,
        mask_file=mask_file,
        save_to=save_to,
    )

# Loading data...
# Loading completed!


In [ ]:
iros_data